# Análisis del indice de calor en Iquitos (1990 - 2025)

**Objetivo:** Identificar en que meses y horas el calor de Iquitos es mas concentrado para poder determinar la demanda de servicios de aire acondicionado.

**Fuente:** NCEI NOAA estacion Aeropuerto Internacional Crnl. FAP Francisco Secada Vignetta.

In [1]:
import pandas as pd

estaciones = pd.read_csv(
    "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv",
    dtype=str
)
estaciones[estaciones["ICAO"].str.contains("SPQT", na=False)]

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
23888,843770,99999,CORONEL FRANCISCO SECADA VIGNETTA INTL,PE,NaN,SPQT,-03.785,-073.309,+0093.3,19730101,20250824


In [2]:
est_iquitos = estaciones[estaciones["ICAO"].str.contains("SPQT", na=False)].iloc[0]
est_iquitos

USAF                                            843770
WBAN                                             99999
STATION NAME    CORONEL FRANCISCO SECADA VIGNETTA INTL
CTRY                                                PE
STATE                                              NaN
ICAO                                              SPQT
LAT                                            -03.785
LON                                           -073.309
ELEV(M)                                        +0093.3
BEGIN                                         19730101
END                                           20250824
Name: 23888, dtype: str

In [3]:
estacion_id = str(est_iquitos['USAF']) + str(est_iquitos["WBAN"])
estacion_id

'84377099999'

In [4]:
partes = []

for anio in range(2000, 2026):
    url = f"https://www.ncei.noaa.gov/data/global-hourly/access/{anio}/{estacion_id}.csv"
    try:
        datos_anio = pd.read_csv(url, usecols=["DATE", "TMP", "DEW"], dtype=str)
        partes.append(datos_anio)
        print(f"{anio}: {len(datos_anio)} registros")
    except Exception:
        print(f"{anio}: sin datos")

df = pd.concat(partes, ignore_index=True)
df.to_csv("../data/raw/iquitos_spqt_2000_2025.csv", index=False)

2000: 9014 registros
2001: 9194 registros
2002: 9237 registros
2003: 9383 registros
2004: 9272 registros
2005: 9302 registros
2006: 9665 registros
2007: 9694 registros
2008: 9692 registros
2009: 9377 registros
2010: 9490 registros
2011: 9561 registros
2012: 9832 registros
2013: 11737 registros
2014: 12475 registros
2015: 12507 registros
2016: 12243 registros
2017: 12222 registros
2018: 12035 registros
2019: 12448 registros
2020: 9844 registros
2021: 11868 registros
2022: 12144 registros
2023: 12090 registros
2024: 12058 registros
2025: 8146 registros


In [5]:
df.head(10)

,DATE,TMP,DEW
0,2000-01-01T00:00:00,"+0294,1","+0241,1"
1,2000-01-01T01:00:00,"+0280,1","+0230,1"
2,2000-01-01T02:00:00,"+0280,1","+0240,1"
3,2000-01-01T03:00:00,"+0270,1","+0240,1"
4,2000-01-01T04:00:00,"+0270,1","+0240,1"
5,2000-01-01T05:00:00,"+0250,1","+0240,1"
6,2000-01-01T06:00:00,"+0250,1","+0236,1"
7,2000-01-01T07:00:00,"+0250,1","+0240,1"
8,2000-01-01T08:00:00,"+0240,1","+0230,1"
9,2000-01-01T09:00:00,"+0230,1","+0230,1"


In [6]:
df.info

<bound method DataFrame.info of                        DATE      TMP      DEW
0       2000-01-01T00:00:00  +0294,1  +0241,1
1       2000-01-01T01:00:00  +0280,1  +0230,1
2       2000-01-01T02:00:00  +0280,1  +0240,1
3       2000-01-01T03:00:00  +0270,1  +0240,1
4       2000-01-01T04:00:00  +0270,1  +0240,1
...                     ...      ...      ...
274525  2025-08-24T19:37:00  +0230,2  +0230,1
274526  2025-08-24T19:43:00  +0230,2  +0230,1
274527  2025-08-24T20:00:00  +0240,2  +0230,1
274528  2025-08-24T21:00:00  +0244,1  +0227,1
274529  2025-08-24T21:00:00  +0250,1  +0230,1

[274530 rows x 3 columns]>

In [7]:
df.shape

(274530, 3)

In [8]:
df.describe()

,DATE,TMP,DEW
count,274530,274530,274530
unique,242588,259,193
top,2017-09-08T00:00:00,"+0240,1","+0230,1"
freq,3,47502,79026


Según la informacion encontrada, podemos decir:
- El año 2025 presenta menos datos porque sus datos llegan solo hasta agosto de ese año. Ademas, los años desde el 2000 al 2010 probablemente la estacion no reportaba las 24 horas.
- Los valores que presentan en `TMP` y `DEW` como `+0275` es a temperatura, pero multiplicada por 10 para evitar decimales. Despues de la coma es el código de calidad, indica que pasó los controles de la NOAA. Ádemas, un valor `+9999` significa dato faltante. 


---

## Decodificar la Temperatura

In [9]:
# 1. Separar el valor y el codigo de calidad
df[["temperatura", "tmp_calidad"]] = df["TMP"].str.split(",", expand=True)
df[["temperatura", "tmp_calidad"]] 

,temperatura,tmp_calidad
0,+0294,1
1,+0280,1
2,+0280,1
3,+0270,1
4,+0270,1
...,...,...
274525,+0230,2
274526,+0230,2
274527,+0240,2
274528,+0244,1


In [10]:
# 2. Convertir de texto a numero
df["temperatura"] = pd.to_numeric(df["temperatura"])
df["temperatura"]

0         294
1         280
2         280
3         270
4         270
         ... 
274525    230
274526    230
274527    240
274528    244
274529    250
Name: temperatura, Length: 274530, dtype: int64

In [11]:
# 3. Marcar los faltantes como vacios
df.loc[df["temperatura"] == 9999, "temperatura" ] = None

In [12]:
# 4. Descartar valores sospechosos y/o erroneos
df.loc[df["tmp_calidad"].isin(["2","3","6","7"]), "temperatura"] = None

In [13]:
# 5. Converir a grados celcius
df["temperatura"] = df["temperatura"] / 10
df["temperatura"]

0         29.4
1         28.0
2         28.0
3         27.0
4         27.0
          ... 
274525     NaN
274526     NaN
274527     NaN
274528    24.4
274529    25.0
Name: temperatura, Length: 274530, dtype: float64

In [14]:
df[["temperatura", "tmp_calidad"]]

,temperatura,tmp_calidad
0,29.4,1
1,28.0,1
2,28.0,1
3,27.0,1
4,27.0,1
...,...,...
274525,NaN,2
274526,NaN,2
274527,NaN,2
274528,24.4,1


---

# Decodificar el Punto de Rocio (DWE)

In [15]:
df["DEW"]

0         +0241,1
1         +0230,1
2         +0240,1
3         +0240,1
4         +0240,1
           ...   
274525    +0230,1
274526    +0230,1
274527    +0230,1
274528    +0227,1
274529    +0230,1
Name: DEW, Length: 274530, dtype: str

In [16]:
df[["punto_rocio", "rocio_calidad"]] = df["DEW"].str.split(",", expand=True)
df[["punto_rocio", "rocio_calidad"]]

,punto_rocio,rocio_calidad
0,+0241,1
1,+0230,1
2,+0240,1
3,+0240,1
4,+0240,1
...,...,...
274525,+0230,1
274526,+0230,1
274527,+0230,1
274528,+0227,1


In [17]:
df["punto_rocio"] = pd.to_numeric(df["punto_rocio"])
df["punto_rocio"]

0         241
1         230
2         240
3         240
4         240
         ... 
274525    230
274526    230
274527    230
274528    227
274529    230
Name: punto_rocio, Length: 274530, dtype: int64

In [18]:
df.loc[df["punto_rocio"] == 9999, "punto_rocio"] = None

In [19]:
df.loc[df["rocio_calidad"].isin(["2","3","6","7"]), "punto_rocio"] = None

In [20]:
df["punto_rocio"] =  df["punto_rocio"] / 10
df["punto_rocio"]

0         24.1
1         23.0
2         24.0
3         24.0
4         24.0
          ... 
274525    23.0
274526    23.0
274527    23.0
274528    22.7
274529    23.0
Name: punto_rocio, Length: 274530, dtype: float64

In [21]:
df[["punto_rocio", "rocio_calidad"]]

,punto_rocio,rocio_calidad
0,24.1,1
1,23.0,1
2,24.0,1
3,24.0,1
4,24.0,1
...,...,...
274525,23.0,1
274526,23.0,1
274527,23.0,1
274528,22.7,1


---
## Fecha y Zona horaria

In [22]:
df["fecha"] = pd.to_datetime(df["DATE"], utc=True).dt.tz_convert("America/Lima")
df["fecha"]

0        1999-12-31 19:00:00-05:00
1        1999-12-31 20:00:00-05:00
2        1999-12-31 21:00:00-05:00
3        1999-12-31 22:00:00-05:00
4        1999-12-31 23:00:00-05:00
                    ...           
274525   2025-08-24 14:37:00-05:00
274526   2025-08-24 14:43:00-05:00
274527   2025-08-24 15:00:00-05:00
274528   2025-08-24 16:00:00-05:00
274529   2025-08-24 16:00:00-05:00
Name: fecha, Length: 274530, dtype: datetime64[us, America/Lima]

In [23]:
df["anio"] = df["fecha"].dt.year
df["anio"]

0         1999
1         1999
2         1999
3         1999
4         1999
          ... 
274525    2025
274526    2025
274527    2025
274528    2025
274529    2025
Name: anio, Length: 274530, dtype: int32

In [24]:
df["hora"] = df["fecha"].dt.hour
df["hora"]

0         19
1         20
2         21
3         22
4         23
          ..
274525    14
274526    14
274527    15
274528    16
274529    16
Name: hora, Length: 274530, dtype: int32

In [25]:
df["fecha_hora"] = df["fecha"].dt.floor("h")
df["fecha_hora"]


0        1999-12-31 19:00:00-05:00
1        1999-12-31 20:00:00-05:00
2        1999-12-31 21:00:00-05:00
3        1999-12-31 22:00:00-05:00
4        1999-12-31 23:00:00-05:00
                    ...           
274525   2025-08-24 14:00:00-05:00
274526   2025-08-24 14:00:00-05:00
274527   2025-08-24 15:00:00-05:00
274528   2025-08-24 16:00:00-05:00
274529   2025-08-24 16:00:00-05:00
Name: fecha_hora, Length: 274530, dtype: datetime64[us, America/Lima]

In [26]:
cobertura = df.groupby("anio")["fecha_hora"].nunique()
cobertura


anio
1999       5
2000    8564
2001    8589
2002    8557
2003    8529
2004    8592
2005    8678
2006    8690
2007    8662
2008    8735
2009    8571
2010    8592
2011    8594
2012    8644
2013    8696
2014    8693
2015    8513
2016    8651
2017    8640
2018    8598
2019    8699
2020    7040
2021    8596
2022    8655
2023    8653
2024    8390
2025    5569
Name: fecha_hora, dtype: int64

In [27]:
cobertura_pct = (cobertura / 8760 * 100).round(1)
cobertura_pct

anio
1999     0.1
2000    97.8
2001    98.0
2002    97.7
2003    97.4
2004    98.1
2005    99.1
2006    99.2
2007    98.9
2008    99.7
2009    97.8
2010    98.1
2011    98.1
2012    98.7
2013    99.3
2014    99.2
2015    97.2
2016    98.8
2017    98.6
2018    98.2
2019    99.3
2020    80.4
2021    98.1
2022    98.8
2023    98.8
2024    95.8
2025    63.6
Name: fecha_hora, dtype: float64

Según los datos, se puede afirmar que: 
- En 1999 presenta 0.1%, es por la zona horaria, por el cambio del UTC.
- En 2025 presenta 63.6% ya que los datos llegan hasta el 24 de agosto.
- En 2020 presenta 80.4%, ya que en ese año sucedio la pandemia y los vuelos nacionales estuvieron suspendidos varios meses y el aeropuerto redujera los reportes

In [28]:
# Verificar la cobertura del año 2020
datos_2020 = df[df["anio"] == 2020]
datos_2020

,DATE,TMP,DEW,temperatura,tmp_calidad,punto_rocio,rocio_calidad,fecha,anio,hora,fecha_hora
208387,2020-01-01T05:00:00,"+0250,1","+0230,1",25.0,1,23.0,1,2020-01-01 00:00:00-05:00,2020,0,2020-01-01 00:00:00-05:00
208388,2020-01-01T06:00:00,"+0245,1","+0232,1",24.5,1,23.2,1,2020-01-01 01:00:00-05:00,2020,1,2020-01-01 01:00:00-05:00
208389,2020-01-01T06:00:00,"+0240,1","+0230,1",24.0,1,23.0,1,2020-01-01 01:00:00-05:00,2020,1,2020-01-01 01:00:00-05:00
208390,2020-01-01T07:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2020-01-01 02:00:00-05:00,2020,2,2020-01-01 02:00:00-05:00
208391,2020-01-01T08:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2020-01-01 03:00:00-05:00,2020,3,2020-01-01 03:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...
218225,2021-01-01T01:00:00,"+0250,1","+0230,1",25.0,1,23.0,1,2020-12-31 20:00:00-05:00,2020,20,2020-12-31 20:00:00-05:00
218226,2021-01-01T02:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2020-12-31 21:00:00-05:00,2020,21,2020-12-31 21:00:00-05:00
218227,2021-01-01T03:00:00,"+0252,1","+0240,1",25.2,1,24.0,1,2020-12-31 22:00:00-05:00,2020,22,2020-12-31 22:00:00-05:00
218228,2021-01-01T03:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2020-12-31 22:00:00-05:00,2020,22,2020-12-31 22:00:00-05:00


In [29]:
datos_2020.groupby(datos_2020["fecha"].dt.month)["fecha_hora"].unique()

fecha
1     [2020-01-01 00:00:00-05:00, 2020-01-01 01:00:0...
2     [2020-02-01 00:00:00-05:00, 2020-02-01 01:00:0...
3     [2020-03-01 00:00:00-05:00, 2020-03-01 01:00:0...
4     [2020-04-01 00:00:00-05:00, 2020-04-01 01:00:0...
5     [2020-05-01 00:00:00-05:00, 2020-05-01 01:00:0...
6     [2020-06-01 00:00:00-05:00, 2020-06-01 07:00:0...
7     [2020-07-01 07:00:00-05:00, 2020-07-01 08:00:0...
8     [2020-08-01 07:00:00-05:00, 2020-08-01 08:00:0...
9     [2020-09-01 07:00:00-05:00, 2020-09-01 08:00:0...
10    [2020-10-01 07:00:00-05:00, 2020-10-01 08:00:0...
11    [2020-11-01 06:00:00-05:00, 2020-11-01 07:00:0...
12    [2020-12-01 00:00:00-05:00, 2020-12-01 01:00:0...
Name: fecha_hora, dtype: object

Se analizan los años 2020 - 2024:
- Se excluirá 1999 al ser un artefacto de la zona horaria no un año real.
- Se excluirá 2025 al estar incompleto
- Se conservará el 2020


In [30]:
df = df[df["anio"].between(2000, 2024)].copy()

In [31]:
df.head()

,DATE,TMP,DEW,temperatura,tmp_calidad,punto_rocio,rocio_calidad,fecha,anio,hora,fecha_hora
5,2000-01-01T05:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2000-01-01 00:00:00-05:00,2000,0,2000-01-01 00:00:00-05:00
6,2000-01-01T06:00:00,"+0250,1","+0236,1",25.0,1,23.6,1,2000-01-01 01:00:00-05:00,2000,1,2000-01-01 01:00:00-05:00
7,2000-01-01T07:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2000-01-01 02:00:00-05:00,2000,2,2000-01-01 02:00:00-05:00
8,2000-01-01T08:00:00,"+0240,1","+0230,1",24.0,1,23.0,1,2000-01-01 03:00:00-05:00,2000,3,2000-01-01 03:00:00-05:00
9,2000-01-01T09:00:00,"+0230,1","+0230,1",23.0,1,23.0,1,2000-01-01 04:00:00-05:00,2000,4,2000-01-01 04:00:00-05:00


---
## Refinamiento 1: solo horas con datos validos

Necesitamos que temperatura y punto de calor sean al mismo tiempo. Filtramos las filas donde ambas columnas tienen valor

In [32]:
validos = df.dropna(subset=["temperatura", "punto_rocio"])
horas_validas = validos.groupby("anio")["fecha_hora"].nunique()

In [33]:
horas_validas

anio
2000    8478
2001    7696
2002    8267
2003    8413
2004    8521
2005    8664
2006    8677
2007    8658
2008    8728
2009    8558
2010    8577
2011    8586
2012    8634
2013    8681
2014    8684
2015    8507
2016    8648
2017    8627
2018    8582
2019    8694
2020    7036
2021    8591
2022    8655
2023    8653
2024    8388
Name: fecha_hora, dtype: int64

## Refinamiento 2: horas reales de cada año

Consideramos los años bisiestos.

In [34]:
horas_anio = pd.Series({
    anio: (366 if pd.Timestamp(year=anio, month=1, day=1).is_leap_year else 365) * 24
    for anio in horas_validas.index
})

Comparamos ambas coberturas

In [35]:
cobertura_real = (horas_validas / horas_anio * 100).round(1)

comparacion = pd.DataFrame({
    "cobertura_inicial": cobertura_pct,
    "cobertura_real": cobertura_real
})
comparacion["diferencia"] = comparacion["cobertura_inicial"] - comparacion["cobertura_real"]
comparacion

,cobertura_inicial,cobertura_real,diferencia
anio,,,
1999,0.1,NaN,NaN
2000,97.8,96.5,1.3
2001,98.0,87.9,10.1
2002,97.7,94.4,3.3
2003,97.4,96.0,1.4
2004,98.1,97.0,1.1
2005,99.1,98.9,0.2
2006,99.2,99.1,0.1
2007,98.9,98.8,0.1


Se investiga que columna falta y en que meses del año 2001

In [36]:
datos_2001 = df[df["anio"] == 2001]
datos_2001[["temperatura", "punto_rocio"]].isna().sum()

temperatura    947
punto_rocio    951
dtype: int64

In [37]:
datos_2001.groupby(datos_2001["fecha"].dt.month)[["temperatura", "punto_rocio"]].count()

,temperatura,punto_rocio
fecha,,
1,795,794
2,688,688
3,781,782
4,765,765
5,432,431
6,514,514
7,583,583
8,763,763
9,737,737


Aplicamos el umbral de 90% de forma estricta: excluimos 2001 y 2020.

In [38]:
anios_validos = cobertura_real[cobertura_real >= 90].index
df = df[df["anio"].isin(anios_validos)].copy()

In [39]:
df

,DATE,TMP,DEW,temperatura,tmp_calidad,punto_rocio,rocio_calidad,fecha,anio,hora,fecha_hora
5,2000-01-01T05:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2000-01-01 00:00:00-05:00,2000,0,2000-01-01 00:00:00-05:00
6,2000-01-01T06:00:00,"+0250,1","+0236,1",25.0,1,23.6,1,2000-01-01 01:00:00-05:00,2000,1,2000-01-01 01:00:00-05:00
7,2000-01-01T07:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2000-01-01 02:00:00-05:00,2000,2,2000-01-01 02:00:00-05:00
8,2000-01-01T08:00:00,"+0240,1","+0230,1",24.0,1,23.0,1,2000-01-01 03:00:00-05:00,2000,3,2000-01-01 03:00:00-05:00
9,2000-01-01T09:00:00,"+0230,1","+0230,1",23.0,1,23.0,1,2000-01-01 04:00:00-05:00,2000,4,2000-01-01 04:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...
266386,2025-01-01T01:00:00,"+0260,1","+0240,1",26.0,1,24.0,1,2024-12-31 20:00:00-05:00,2024,20,2024-12-31 20:00:00-05:00
266387,2025-01-01T02:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2024-12-31 21:00:00-05:00,2024,21,2024-12-31 21:00:00-05:00
266388,2025-01-01T03:00:00,"+0250,1","+0238,1",25.0,1,23.8,1,2024-12-31 22:00:00-05:00,2024,22,2024-12-31 22:00:00-05:00
266389,2025-01-01T03:00:00,"+0250,1","+0240,1",25.0,1,24.0,1,2024-12-31 22:00:00-05:00,2024,22,2024-12-31 22:00:00-05:00
